# Foundation LPS Geometry Validation

Validates the explicit LPS contract across versioned predrr volumes, four-channel targets and DRR inputs. Missing versioned outputs remain pending rather than being treated as successful.

Set REQUIRE_ALL_READY to True only for the final Stage 1 gate.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import nibabel as nib

ROOT = Path.cwd()
while not (ROOT / "configs").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

CONTRACT = json.loads((ROOT / "configs/data_contract_v1.json").read_text())
MANIFEST = ROOT / "reports/manifests/quantitative_manifest_v1.csv"
REPORT = ROOT / "reports/manifests/lps_geometry_validation_v1.csv"
REQUIRE_ALL_READY = False
BONES = CONTRACT["bone_channels"]
EXPECTED_AXCODES = ("L", "P", "S")
EXPECTED_SHAPE = tuple(CONTRACT["final_shape"])
EXPECTED_SPACING = np.asarray(CONTRACT["final_spacing_mm"])

In [ ]:
def assert_lps(image):
    assert nib.aff2axcodes(image.affine) == EXPECTED_AXCODES
    assert image.shape == EXPECTED_SHAPE
    assert np.allclose(image.header.get_zooms()[:3], EXPECTED_SPACING, atol=1e-6)

affine = np.diag([-0.78125, -0.78125, 0.78125, 1.0])
synthetic = nib.Nifti1Image(np.zeros(EXPECTED_SHAPE, dtype=np.uint8), affine)
assert_lps(synthetic)

landmark_voxel = np.array([10.0, 20.0, 30.0])
world = nib.affines.apply_affine(affine, landmark_voxel)
assert world[0] < 0 and world[1] < 0 and world[2] > 0

swapped_affine = affine.copy()
swapped_affine[:, [0, 1]] = swapped_affine[:, [1, 0]]
assert nib.aff2axcodes(swapped_affine) != EXPECTED_AXCODES
print("SYNTHETIC LPS AND INTENTIONAL AXIS-SWAP TEST PASS")

In [ ]:
manifest = pd.read_csv(MANIFEST)
candidates = manifest[manifest.status != "excluded"].copy()

def check_sample(row):
    result = {
        "sample_id": row.sample_id,
        "status": row.status,
        "predrr": "missing",
        "target": "missing",
        "drr_pair": "missing",
        "result": "pending",
    }
    predrr = ROOT / row.predrr_path
    target_dir = ROOT / row.target_path
    ap, lat = ROOT / row.ap_drr_path, ROOT / row.lat_drr_path
    if not predrr.exists():
        return result

    predrr_image = nib.load(str(predrr))
    assert_lps(predrr_image)
    result["predrr"] = "pass"

    target_files = [target_dir / f"{row.sample_id}_{bone}.nii.gz" for bone in BONES]
    if not all(path.exists() for path in target_files):
        return result

    for path in target_files:
        target_image = nib.load(str(path))
        assert_lps(target_image)
        assert np.allclose(target_image.affine, predrr_image.affine, atol=1e-5)
        assert np.asarray(target_image.dataobj).max() > 0
    result["target"] = "pass"

    if ap.exists() and lat.exists():
        ap_array, lat_array = np.load(ap), np.load(lat)
        assert ap_array.shape == (256, 256)
        assert lat_array.shape == (256, 256)
        assert np.isfinite(ap_array).all() and np.isfinite(lat_array).all()
        result["drr_pair"] = "pass"

    result["result"] = "pass" if all(result[key] == "pass" for key in ["predrr", "target", "drr_pair"]) else "pending"
    return result

rows = pd.DataFrame(check_sample(row) for row in candidates.itertuples(index=False))
REPORT.parent.mkdir(parents=True, exist_ok=True)
rows.to_csv(REPORT, index=False)
display(rows.groupby("result").size().rename("samples"))
if REQUIRE_ALL_READY:
    assert len(rows) == 71
    assert (rows.result == "pass").all(), rows[rows.result != "pass"]
print("geometry report:", REPORT)

## HPC result request

After a full versioned preprocessing or DRR batch, return the executed notebook or log, preprocessing metadata, DRR metadata, this geometry report, AP/LAT overlays, job ID, resource usage, output paths and hashes. The batch is not certified until reviewed and marked PASS.